# Experiment 003 (train-distribution dependence) on Google Colab Pro+ (A100)
Run the cells top to bottom. Cells 1-8 are idempotent: after a disconnect choose a GPU runtime and run them again.

**READ FIRST: if the session is cut, the run that was training is trained again from epoch 0 (about 3 hours per run, estimate, not measured).**
This notebook deliberately has NO resume patch (the 002 `train_resume.patch` is not used): continuing from a checkpoint would not be bit-identical, and that is a user decision.
Runs that have a `DONE` marker are skipped, so after a disconnect only the interrupted run is lost. Re-run the training cell (same `EXP003_APPROVED`) and it continues with the next unfinished run.

**Human approval (CLAUDE.md): cell 9 (smoke) and cell 10 (training) use the GPU. Do not switch `RUN_SMOKE` / `RUN_TRAIN` to True, and do not fill `EXP003_APPROVED`, before you have approved it.**
The training cell refuses to start when `EXP003_APPROVED` is empty, and `run_all.sh` also refuses without a finished `results/smoke.json`.

Session tips (Colab behaviour from memory, 추정, not checked today)
- Colab may reclaim an idle runtime (about 90 min without interaction) and has a maximum session length (about 24 h). Keep the browser tab open; Pro+ background execution is not a guarantee.
- Everything that must survive lives on Drive in `MyDrive/exp003_work`: `train_runs/` (checkpoints, DONE), `exp003_cache/`, `results/`, `logs/`, `manifests/`. `/content` (venv, code, copied data) is rebuilt every session.
- Long commands run in the background (nohup, log file); the cell only tails the log. Interrupting a cell does not stop the background job.
- Code delivery: the 003 folder is NOT on GitHub. Cell 3 unpacks `MyDrive/exp003_work/exp003_code.tar` if it exists (takes priority), otherwise the snapshot embedded in cell 3 (generated by `colab/build_notebook.py`; regenerate the notebook after code changes).
- Everything in this notebook is UNVERIFIED on Colab as a whole (only the individual facts listed in the README were observed).

In [1]:
# 1. GPU / Python check (+ helpers used by later cells)
import sys, subprocess, os, time
def sh(cmd, cwd=None):
    # run a shell command and stream its output into the cell
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

def bg(cmd, log, cwd=None, env=None, every=30):
    # run cmd with nohup in its own session, output to `log`; tail the log until it ends. Returns the exit code.
    # Interrupting the cell leaves the job running (re-run the cell: the commands used here skip finished work).
    os.makedirs(os.path.dirname(log), exist_ok=True)
    rcfile = log + '.rc'
    if os.path.exists(rcfile):
        os.remove(rcfile)
    full = '(%s) >> %s 2>&1; echo $? > %s' % (cmd, log, rcfile)
    subprocess.Popen(['nohup', 'bash', '-c', full], cwd=cwd, env=env, start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    shown = 0
    try:
        while not os.path.exists(rcfile):
            time.sleep(every)
            lines = open(log, errors='replace').read().splitlines() if os.path.exists(log) else []
            print('\n'.join(lines[shown:][-15:]), flush=True)
            shown = len(lines)
    except KeyboardInterrupt:
        print('cell interrupted; the background job keeps running (log: %s)' % log)
        return None
    lines = open(log, errors='replace').read().splitlines()
    print('\n'.join(lines[shown:][-30:]))
    return int(open(rcfile).read().strip() or 1)

sh('nvidia-smi')
print(sys.version)
print(subprocess.run("df -h /content | tail -1", shell=True, capture_output=True, text=True).stdout)

Wed Nov  4 12:05:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2. Mount Drive and choose the persistent work root
import os
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT = os.environ.get('EXP003_WORK_ROOT', '/content/drive/MyDrive/exp003_work')
os.makedirs(WORK_ROOT, exist_ok=True)
# Data source folder (names contain brackets and a space: only passed through os.environ / pathlib, never through a shell command line).
DATA_SRC = os.environ.get('EXP003_DRIVE_MOUNT_SRC', '/content/drive/MyDrive/dino-wm-videos/[data]/[cloth-sim] cloth_move')
print('WORK_ROOT =', WORK_ROOT)
print('archive cloth_sim_data.tar on Drive:', os.path.exists(WORK_ROOT + '/cloth_sim_data.tar'))
print('mount source exists:', os.path.isdir(DATA_SRC))

Mounted at /content/drive
WORK_ROOT = /content/drive/MyDrive/exp003_work
archive cloth_sim_data.tar on Drive: True
mount source exists: True


In [3]:
# 3. Unpack the 003 code to /content/mylab/experiments/003_train_dist_dependence
# Priority: WORK_ROOT/exp003_code.tar (any tar / tar.gz) if present, else the snapshot embedded below.
# The embedded snapshot was generated at notebook build time (2026-11-04 11:30 UTC); when the code changes, re-run `python3 colab/build_notebook.py` and re-upload the notebook.
# Only code files are overwritten; results/ logs/ manifests/ (Drive symlinks, cell 6) are untouched.
import base64, io, os, tarfile
TOP = '003_train_dist_dependence'
REPO = '/content/mylab'
EXP = REPO + '/experiments/' + TOP
EMBEDDED = """
...
"""
ext_tar = WORK_ROOT + '/exp003_code.tar'
if os.path.exists(ext_tar):
    src = open(ext_tar, 'rb').read()
    print('using', ext_tar, len(src), 'bytes')
else:
    src = base64.b64decode(''.join(EMBEDDED.split()))
    print('using the embedded snapshot', len(src), 'bytes')
os.makedirs(os.path.dirname(EXP), exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(src), mode='r:*') as tf:
    for m in tf.getmembers():
        parts = m.name.split('/')
        assert not m.name.startswith('/') and '..' not in parts and parts[0] == TOP and (m.isfile() or m.isdir()), 'refused tar member: ' + m.name
    tf.extractall(os.path.dirname(EXP))
os.chmod(EXP + '/run_all.sh', 0o755)
assert os.path.isfile(EXP + '/run.py') and os.path.isfile(EXP + '/colab/requirements_colab.txt')
print('code in', EXP)

using the embedded snapshot 63348 bytes
code in /content/mylab/experiments/003_train_dist_dependence


/tmp/ipykernel_611/1292469246.py:868: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(os.path.dirname(EXP))


In [4]:
# 4. Environment: uv venv with Python 3.9 (no conda on Colab; documented exception to CLAUDE.md). Rebuilt every new session (a few minutes).
# Verified in an earlier session: after install, import pkg_resources,torch,numpy,cv2,decord,hydra passes; torch 2.0.1+cu117, cuda True.
import hashlib, os
VENV = '/content/venv'
PY = VENV + '/bin/python'
MARK = VENV + '/.exp003_ok_' + hashlib.sha256(open(EXP + '/colab/requirements_colab.txt', 'rb').read()).hexdigest()[:12]
if not os.path.exists(MARK):
    assert sh('pip install -q uv') == 0
    if not os.path.exists(PY):
        assert sh('uv venv --python 3.9 %s' % VENV) == 0
    assert sh('uv pip install --python %s --index-strategy unsafe-best-match -r %s/colab/requirements_colab.txt' % (PY, EXP)) == 0
    open(MARK, 'w').write('ok\n')
CHK = "import sys,pkg_resources,torch,numpy,cv2,decord,hydra;print(sys.version.split()[0],'torch',torch.__version__,'cuda',torch.version.cuda,torch.cuda.is_available(),'numpy',numpy.__version__)"
assert sh('%s -c "%s"' % (PY, CHK)) == 0, 'environment check failed'

Using CPython 3.9.25
Creating virtual environment at: venv
Activate with: source venv/bin/activate
Using Python 3.9.25 environment at: venv
Resolved 55 packages in 1.80s
Prepared 55 packages in 19.93s
Installed 55 packages in 186ms
 + accelerate==0.26.1
 + antlr4-python3-runtime==4.9.3
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.1.8
 + cloudpickle==3.1.2
 + cmake==4.4.4
 + decord==0.6.0
 + docker-pycreds==0.4.0
 + einops==0.4.1
 + filelock==3.19.1
 + fsspec==2025.10.0
 + gitdb==4.0.12
 + gitpython==3.2.1
 + huggingface-hub==0.23.2
 + hydra-core==1.2.0
 + hydra-submitit-launcher==1.2.0
 + idna==3.20
 + imageio==2.37.2
 + jinja2==3.1.6
 + lit==23.1.5
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.2.1
 + numpy==1.26.4
 + omegaconf==2.3.0
 + opencv-python==4.6.0.66
 + packaging==26.3
 + pathtools==0.1.2
 + pillow==10.3.0
 + promise==2.3
 + protobuf==3.20.3
 + psutil==5.9.8
 + pywavelets==1.6.0
 + pyyaml==6.0.3
 + requests==2.32.5
 + safetensors==0.7.0
 + scikit-im

In [5]:
# 5. ACWM-Phys checkout at the pinned commit (== config.yaml code.commit). No patch is applied (no resume patch in 003).
import os, subprocess
COMMIT = '5a640bed66002eca7bdb31d10b2bde59ad944551'
SRC = '/content/dinowm/acwm_phys_src'
if not os.path.isdir(SRC + '/.git'):
    os.makedirs(os.path.dirname(SRC), exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/jongmin-s-multicore/thread0.git', SRC], check=True)
subprocess.run(['git', '-C', SRC, 'checkout', '--detach', COMMIT], check=True)
head = subprocess.run(['git', '-C', SRC, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, head
dirty = subprocess.run(['git', '-C', SRC, 'status', '--porcelain', '--untracked-files=no'], capture_output=True, text=True).stdout.strip()
assert not dirty, 'acwm_phys_src has local modifications: ' + dirty
print('acwm-phys at', head)

acwm-phys at 5a640bed66002eca7bdb31d10b2bde59ad944551


In [6]:
# 6. Environment variables + Drive-backed results/logs/manifests (symlinks into WORK_ROOT)
import os, shutil
os.environ.update({
    'DINO_WORK': '/content/dinowm',
    'DATASET_DIR': '/content/dinowm/data',                 # local copy (fast reads)
    'DINO_TRAIN': WORK_ROOT + '/train_runs',               # Drive: checkpoints, DONE markers
    'EXP003_CACHE': WORK_ROOT + '/exp003_cache',           # Drive
    'ACWM_PHYS_SRC': SRC,
    'WANDB_MODE': 'disabled',
    'EXP003_WORK_ROOT': WORK_ROOT,                         # archive lookup/creation: WORK_ROOT/cloth_sim_data.tar
    'EXP003_DATA_MODE': 'mount',
    'CONDA_ENV': 'none',                                   # run_all.sh: no conda on Colab
    'PATH': os.path.dirname(PY) + os.pathsep + os.environ['PATH'],
    'VIRTUAL_ENV': VENV,
})
# NOTE: PYTORCH_CUDA_ALLOC_CONF is deliberately NOT set here. run.py adds expandable_segments only for torch >= 2.1 and respects a value set by you.
if DATA_SRC:
    os.environ['EXP003_DRIVE_MOUNT_SRC'] = DATA_SRC
for d in ('train_runs', 'exp003_cache'):
    os.makedirs(WORK_ROOT + '/' + d, exist_ok=True)
for d in ('results', 'logs', 'manifests'):
    tgt = WORK_ROOT + '/' + d
    lnk = EXP + '/' + d
    os.makedirs(tgt, exist_ok=True)
    if os.path.islink(lnk):
        if os.path.realpath(lnk) != os.path.realpath(tgt):
            os.remove(lnk)
    if not os.path.islink(lnk):
        if os.path.isdir(lnk):                              # real folder: merge into Drive (never overwrite), then replace by a link
            shutil.copytree(lnk, tgt, dirs_exist_ok=True)
            shutil.rmtree(lnk)
        os.symlink(tgt, lnk)
    print(lnk, '->', os.path.realpath(lnk))

/content/mylab/experiments/003_train_dist_dependence/results -> /content/drive/MyDrive/exp003_work/results
/content/mylab/experiments/003_train_dist_dependence/logs -> /content/drive/MyDrive/exp003_work/logs
/content/mylab/experiments/003_train_dist_dependence/manifests -> /content/drive/MyDrive/exp003_work/manifests


In [7]:
# 7. Data -> /content/dinowm/data (command: python run.py download). Idempotent: complete splits are skipped.
#    a) WORK_ROOT/cloth_sim_data.tar (1.45 GB, 16 splits) exists -> extract it locally.
#    b) otherwise parallel copy from the mounted Drive folder (about 10 min; 0 warnings in an earlier check), then the archive is written once to WORK_ROOT.
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)
rc = bg('%s run.py download' % PY, EXP + '/logs/colab_download.log', cwd=EXP)
assert rc == 0, 'download failed or cell interrupted (rc=%s); read the lines above, then re-run this cell' % rc

overlay         236G  4.3G  232G   2% /

[2026-11-04 11:47:37]   [ind_train] done: 2000 episodes, 0.19 GB (source: archive)
[2026-11-04 11:47:38]     a125_train: 2001/2001 files, 0.21 GB, 1 s (finished)
[2026-11-04 11:47:38]   [a125_train] done: 2000 episodes, 0.21 GB (source: archive)
[2026-11-04 11:47:38]     a040_train: 2001/2001 files, 0.13 GB, 0 s (finished)
[2026-11-04 11:47:39]   [a040_train] done: 2000 episodes, 0.13 GB (source: archive)
[2026-11-04 11:47:39]     a160_train: 2001/2001 files, 0.24 GB, 1 s (finished)
[2026-11-04 11:47:40]   [a160_train] done: 2000 episodes, 0.24 GB (source: archive)
[2026-11-04 11:47:43] download check: 16 splits, 0 warnings -> /content/mylab/experiments/003_train_dist_dependence/results/download_check.json
[2026-11-04 12:08:54] log file: /content/mylab/experiments/003_train_dist_dependence/logs/download_20261104-120854.log
[2026-11-04 12:08:54] WARNING: CONDA_DEFAULT_ENV is empty - not inside an activated conda environment (python=/content/venv/

In [8]:
# 8. check / selftest / maskcheck / listing (no GPU training; each stage is idempotent)
for stage in ('check', 'selftest', 'maskcheck', 'listing'):
    print('=== run.py %s' % stage)
    rc = bg('%s run.py %s' % (PY, stage), EXP + '/logs/colab_%s.log' % stage, cwd=EXP, every=15)
    assert rc == 0, 'run.py %s failed (rc=%s); see %s/logs' % (stage, rc, EXP)

=== run.py check
    "single_a026:0": false,
    "single_a026:1": false,
    "single_a026:2": false,
    "single_a248:0": false,
    "single_a248:1": false,
    "single_a248:2": false,
    "mix_adjacent:0": false,
    "mix_adjacent:1": false,
    "mix_adjacent:2": false,
    "mix_wide:0": false,
    "mix_wide:1": false,
    "mix_wide:2": false
  },
  "approved_distributions": []
}

=== run.py selftest
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x59bc76b4e740] moov atom not found
[2026-11-04 11:43:18]   WARNING ind_train: first video not probed (cannot open)
[2026-11-04 11:43:18]   WARNING ind_test: first video not probed (cannot open)
[2026-11-04 11:43:18] download check: 2 splits, 2 warnings -> /tmp/tmpzqibnzqw/C/results/download_check.json
[2026-11-04 11:43:18] data source: archive /tmp/tmpzqibnzqw/work/cloth_sim_data.tar (one file; fast path)
[2026-11-04 11:43:18] extract_archive: /tmp/tmpzqibnzqw/work/cloth_sim_data.tar -> /tmp/tmpzqibnzqw/D/data/cloth_sim splits to extract=['ind_train', 'ind_test

In [16]:
# 9. SMOKE (GPU run: 12 GB memory/time measurement). Needs human approval (CLAUDE.md): set RUN_SMOKE = True only after approval.
RUN_SMOKE = True
if RUN_SMOKE:
    if os.path.exists(EXP + '/results/smoke.json'):
        print('results/smoke.json exists; showing it (delete it on Drive to repeat the smoke)')
        print(open(EXP + '/results/smoke.json').read()[:3000])
    else:
        rc = bg('%s run.py smoke' % PY, EXP + '/logs/colab_smoke.log', cwd=EXP, every=30)
        print('smoke rc =', rc)
else:
    print('RUN_SMOKE is False: skipped')

Epoch 1 Train: 100%|██████████| 124/124 [01:08<00:00,  1.81it/s]

Epoch 1 Valid:   0%|          | 0/14 [00:00<?, ?it/s]
proprio_dim: 10, after repeat: 10
action_dim: 10, after repeat: 10
emb_dim: 404
Model emb_dim:  404
[2026-11-04 12:16:25] loaded exp003_mix_wide_s0_smoke epoch 1; num_hist=1 frameskip=5
[2026-11-04 12:16:25] episodes to extract: 45  groups: {'a026': 5, 'a040': 5, 'a060': 5, 'a080': 5, 'ind': 5, 'a125': 5, 'a160': 5, 'a200': 5, 'a248': 5}
/content/venv/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/content/venv/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of 

In [17]:
# 10. FULL TRAINING (GPU, about 3 h per run, estimate). Needs human approval (CLAUDE.md).
# Fill EXP003_APPROVED with the distribution names YOU approved (space separated, e.g. 'single_a026 single_a248 mix_adjacent mix_wide'). Empty -> refuses.
# Resume behaviour: run_all.sh skips every run that has DONE (WORK_ROOT/train_runs/outputs/<run>/DONE). After a disconnect re-run this cell: finished runs
# are skipped, the interrupted run is trained again from epoch 0 (no resume patch). Stages after train (extract/metrics/aggregate) also run, per model.
RUN_TRAIN = True
EXP003_APPROVED = 'mix_wide'
if RUN_TRAIN:
    approved = EXP003_APPROVED.split()
    assert approved, 'EXP003_APPROVED is empty: write the approved distribution names yourself. Refusing to train.'
    assert os.path.exists(EXP + '/results/smoke.json') and '"status": "finished"' in open(EXP + '/results/smoke.json').read(), \
        'results/smoke.json missing or not finished: run the smoke (cell 9) first'
    import json, subprocess
    env = dict(os.environ, EXP003_APPROVED=' '.join(approved))
    models = subprocess.run([PY, '-c', "import sys;sys.path.insert(0,'src');from exp003.common import load_config,all_models;print(' '.join(all_models(load_config())))"],
                            cwd=EXP, env=env, capture_output=True, text=True, check=True).stdout.split()
    env['MODELS'] = ' '.join(m for m in models if m.split(':')[0] in approved)         # only approved distributions (run_all.sh stops at a non-approved one)
    print('models to run:', env['MODELS'])
    assert env['MODELS'], 'none of the approved names match config distributions: ' + ' '.join(models)
    LOG = EXP + '/logs/train_all.log'
    running = subprocess.run('pgrep -f "[r]un_all.sh"', shell=True, capture_output=True).returncode == 0
    if running:
        print('run_all.sh is already running in this session; attaching to its log')
    else:
        p = subprocess.Popen('bash run_all.sh --from train >> %s 2>&1' % LOG, shell=True, cwd=EXP, env=env, start_new_session=True)
        print('started run_all.sh pid', p.pid)
    try:
        while subprocess.run('pgrep -f "[r]un_all.sh"', shell=True, capture_output=True).returncode == 0:
            print(subprocess.run(['tail', '-n', '3', LOG], capture_output=True, text=True).stdout, flush=True)
            time.sleep(300)          # keeps the cell busy; interrupting the cell does not stop the background job
    except KeyboardInterrupt:
        print('cell interrupted; the background job keeps running')
    print(subprocess.run(['tail', '-n', '20', LOG], capture_output=True, text=True).stdout)
else:
    print('RUN_TRAIN is False: skipped')

models to run: mix_wide:0 mix_wide:1 mix_wide:2
started run_all.sh pid 22597

emb_dim: 404
Model emb_dim:  404

Epoch 1 Train:  28%|██▊       | 1151/4125 [04:51<12:10,  4.07it/s]
emb_dim: 404
Model emb_dim:  404

Epoch 1 Train:  58%|█████▊    | 2373/4125 [09:50<07:11,  4.06it/s]
emb_dim: 404
Model emb_dim:  404

Epoch 1 Train:  87%|████████▋ | 3596/4125 [14:51<02:08,  4.11it/s]

Epoch 1 Valid: 100%|██████████| 459/459 [01:20<00:00,  5.72it/s]
[2026-11-04 12:35:27,908][__main__][INFO] - Epoch 1  Training loss: 1.9473                  Validation loss: 1.5318

Epoch 2 Train:   6%|▋         | 262/4125 [01:12<15:47,  4.08it/s]

Epoch 1 Valid: 100%|██████████| 459/459 [01:20<00:00,  5.72it/s]
[2026-11-04 12:35:27,908][__main__][INFO] - Epoch 1  Training loss: 1.9473                  Validation loss: 1.5318

Epoch 2 Train:  36%|███▌      | 1485/4125 [06:12<10:52,  4.04it/s]

Epoch 1 Valid: 100%|██████████| 459/459 [01:20<00:00,  5.72it/s]
[2026-11-04 12:35:27,908][__main__][INFO] - Epoch 1  T

: 

In [ ]:
# 11. After everything is finished (or before you leave): flush Drive writes.
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
# Status monitor (last cell). One block every 3 min, also appended to Drive (exp003_work/heartbeat.log). Interrupting it does not stop training.
def monitor(every=180):
    import glob, json, os, subprocess, time
    mount = '/content/drive'
    work = mount + '/MyDrive/exp003_work'

    def out(cmd):
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

    def alive(pattern):
        return 'ON' if out('pgrep -f "%s"' % pattern) else 'off'

    def runs():
        rows = []
        for d in sorted(glob.glob(work + '/train_runs/outputs/*')):
            name = os.path.basename(d)
            if os.path.exists(d + '/DONE'):
                rows.append(name + ' DONE')
                continue
            p = d + '/epoch_logs.jsonl'      # written by the pinned train.py (name as in 002; absent -> "no epoch yet")
            try:
                with open(p) as f:
                    last = json.loads(f.read().strip().splitlines()[-1])
                rows.append('%s e%s val_loss=%s (saved %d min ago)' % (name, last.get('epoch'), last.get('val_loss'), (time.time() - os.path.getmtime(p)) / 60))
            except (OSError, IndexError, ValueError):
                rows.append(name + ' started, no epoch finished yet')
        return ' | '.join(rows) or '(no run folder visible)'

    def last_line():
        try:
            lines = open(work + '/logs/train_all.log', errors='replace').read().splitlines()
            return lines[-1][:200] if lines else '(empty log)'
        except OSError as e:
            return '(cannot read train_all.log: %s)' % e.strerror

    def status():
        gpu = out('nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader') or 'n/a'
        text = '[%s UTC] run_all=%s train=%s gpu=%s\n    %s\n    %s' % (
            time.strftime('%H:%M', time.gmtime()), alive('[r]un_all.sh'), alive('[t]rain.py --config-name'), gpu, runs(), last_line())
        print(text, flush=True)
        if not os.path.ismount(mount):
            print('    !!! DRIVE IS NOT MOUNTED: training cannot save. Do not wait, paste this output.', flush=True)
            return
        try:
            with open(work + '/heartbeat.log', 'a') as f:
                f.write(text + '\n')
        except OSError as e:
            print('    heartbeat not written: %s' % e.strerror, flush=True)

    try:
        while alive('[r]un_all.sh') == 'ON':
            status()
            time.sleep(every)
        status()
        print('run_all.sh is NOT running: finished or stopped. Read the last line above.')
    except KeyboardInterrupt:
        print('monitor stopped; training keeps running')

monitor()